In [ ]:
!pip install datasets
!pip install sentence-transformers
!pip install faiss-cpu
!pip install gradio
!pip install groq
!pip install langchain-text-splitters
!pip install scikit-learn


In [ ]:
from datasets import load_dataset

from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
)

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from groq import Groq

import numpy as np
import faiss
import gradio as gr

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from groq import Groq

import numpy as np
import faiss
import gradio as gr

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "deokhk/ar_wiki_sentences_1000000",
    split="train[:30000]"
)

print(dataset)

In [ ]:
print(dataset.column_names)

In [ ]:
texts = []

for item in dataset:

    text = item["sentence"]
    text = str(text).strip()

    if len(text) > 50:

        texts.append(text)

print("Total Texts:", len(texts))

print(texts[0])

In [ ]:
embedding_model = SentenceTransformer(
    "intfloat/multilingual-e5-base"
)

In [ ]:
def retrieve(query, k=3):

    query_embedding = embedding_model.encode([query])

    xq = np.array(query_embedding).astype("float32")

    distance, pos = index.search(xq, k)

    results = [all_chunks[i] for i in pos[0]]

    return results

In [ ]:
def fixed_chunking(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i+chunk_size]
        chunks.append(chunk)

    return chunks

In [ ]:
def recursive_chunking(text):

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=120,
        chunk_overlap=30,
        separators=["\n\n", "\n", ". ", " "]
    )

    return splitter.split_text(text)

In [ ]:
def sentence_chunking(text):

    sentences = text.split(".")

    chunks = []

    current = ""

    for sentence in sentences:

        if len(current) + len(sentence) < 500:
            current += sentence + "."
        else:
            chunks.append(current)
            current = sentence + "."

    if current:
        chunks.append(current)

    return chunks

In [ ]:


def semantic_chunking(text, threshold=0.5):

    sentences = text.split(".")

    sentences = [
        s.strip()
        for s in sentences
        if s.strip()
    ]

    if len(sentences) == 0:
        return []

    # embeddings
    embeddings = embedding_model.encode(sentences)

    chunks = []

    current_chunk = [sentences[0]]

    for i in range(1, len(sentences)):

        similarity = cosine_similarity(
            [embeddings[i - 1]],
            [embeddings[i]]
        )[0][0]

        # chunk
        if similarity >= threshold:

            current_chunk.append(sentences[i])

        # → chunk
        else:

            chunks.append(". ".join(current_chunk))

            current_chunk = [sentences[i]]

    # آخر chunk
    chunks.append(". ".join(current_chunk))

    return chunks

In [ ]:
def clean_chunk(chunk):

    if isinstance(chunk, list):
        chunk = " ".join(chunk)

    return str(chunk).strip()

In [ ]:
all_chunks = []

for text in texts:

    chunks = recursive_chunking(text)

    for chunk in chunks:

        chunk = str(chunk).strip()

        if len(chunk) > 20:

            all_chunks.append(chunk)

print("Total Chunks:", len(all_chunks))

In [ ]:
embeddings = embedding_model.encode(all_chunks)

In [ ]:
import torch
X = torch.rand(1024, 50_000, dtype=torch.float32)
X.shape

In [ ]:
import faiss

index_dim = X.shape[1]

index = faiss.IndexFlatL2(index_dim)

index.add(X)

print(index.ntotal)

In [ ]:
X = np.array(embeddings).astype("float32")

index = faiss.IndexFlatL2(X.shape[1])

index.add(X)

In [ ]:
import os

client = Groq(
    ...=os.environ["..."]
)

In [ ]:
def rag_chat(question):

    retrieved_chunks = retrieve(question)

    context = "\n".join(retrieved_chunks)

    prompt = f"""
    أجب اعتمادًا على المعلومات التالية فقط:

    {context}

    السؤال:
    {question}
    """

    response = client.chat.completions.create(

        model="llama-3.1-8b-instant",

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0.3,

        max_tokens=500
    )

    return response.choices[0].message.content

In [ ]:
def retrieve(query, k=3):

    query_embedding = embedding_model.encode([query])

    xq = np.array(query_embedding).astype("float32")

    distance, pos = index.search(xq, k)

    results = [all_chunks[i] for i in pos[0]]

    return results

In [ ]:
answer = rag_chat(
      "لاعب كرة قدم بريطاني"
 )

print(answer)

In [ ]:
import gradio as gr

interface = gr.Interface(

    fn=rag_chat,

    inputs="text",

    outputs="text",

    title="Arabic RAG Chatbot"
)

interface.launch()